# Restaurant Review Sentiment

Binary **positive vs negative** classification on real restaurant reviews.

**Use case:** Monitor product or dining feedback at scale.

**Prerequisites:** `01-nlp-fundamentals.ipynb` introduces the concepts. This notebook uses `nlp_helpers.py` for preprocessing.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

from nlp_helpers import DATASETS_DIR, download_nltk_data, preprocess_text

download_nltk_data()
print('Setup complete.')


## 1. Load the dataset


In [ ]:
path = f'{DATASETS_DIR}/restaurant-reviews.tsv'
df = pd.read_csv(path, sep='\t', quoting=3)
df.columns = ['Review', 'Liked']  # 1 = liked, 0 = not
print('Shape:', df.shape)
print(df['Liked'].value_counts())
df.head()


## 2. Train a classifier

Pipeline: `preprocess_text` → TF-IDF (unigrams + bigrams) → logistic regression.


In [ ]:
df['processed'] = df['Review'].apply(preprocess_text)

tfidf = TfidfVectorizer(max_features=1500, ngram_range=(1, 2))
X = tfidf.fit_transform(df['processed'])
y = df['Liked']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print('Accuracy:', round(accuracy_score(y_test, y_pred), 3))
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))
print('Confusion matrix:\n', confusion_matrix(y_test, y_pred))


## 3. Interpret coefficients

Large positive weights → predictive of **liked** reviews.


In [ ]:
feature_names = tfidf.get_feature_names_out()
coef = clf.coef_[0]
top_positive = sorted(zip(feature_names, coef), key=lambda x: x[1], reverse=True)[:10]
top_negative = sorted(zip(feature_names, coef), key=lambda x: x[1])[:10]
print('Top words for POSITIVE:')
for w, c in top_positive:
    print(f'  {w}: {c:.3f}')
print('\nTop words for NEGATIVE:')
for w, c in top_negative:
    print(f'  {w}: {c:.3f}')
